# Q21: Optimize Prediction Throughput for RNN
**Topic:** NLP | **Difficulty:** Medium | **Time:** 15 min
**Sheet:** Grind75ML

---

## Question
How can you optimize the prediction throughput for an RNN?

## Detailed Answer

### 1. Batching
Process multiple sequences simultaneously instead of one at a time.
- **Padding**: Pad shorter sequences to the length of the longest in the batch
- **Bucketing**: Group similar-length sequences to minimize padding waste
- **Dynamic batching**: Collect incoming requests and batch them

### 2. Sequence Length Optimization
- **Truncation**: Limit max sequence length (often 95th percentile of training data)
- **Packed sequences**: PyTorch's `pack_padded_sequence` skips computation on padded tokens

### 3. Model Optimization
| Technique | Speedup | Quality Loss |
|-----------|---------|-------------|
| **Quantization** (INT8) | 2-4x | Minimal |
| **Pruning** (structured) | 2-3x | Minor |
| **Knowledge Distillation** | Custom | Depends |
| **ONNX/TensorRT** | 2-5x | None |

### 4. CUDA Optimizations
- **cuDNN RNN**: Use optimized CUDA RNN kernels (much faster than manual loop)
- **FP16**: Mixed-precision inference halves memory bandwidth
- **Persistent RNNs**: Keep weights in registers for small models

### 5. Architectural Changes
- **Replace RNN with Transformer**: Self-attention is parallelizable across time
- **Replace RNN with 1D CNN**: TCN (Temporal Conv Net) parallelizes fully
- **Use GRU instead of LSTM**: Fewer gates, fewer parameters, similar performance
- **Reduce hidden size**: Smaller hidden state = faster computation

### 6. Infrastructure
- **Model serving**: Triton, TorchServe with dynamic batching
- **Horizontal scaling**: Multiple model replicas behind a load balancer
- **Caching**: Cache predictions for repeated inputs
- **Async inference**: Non-blocking calls for non-real-time requests

### 7. Early Exit / Conditional Computation
- Add classifiers at intermediate time steps
- If confidence threshold is met early, return prediction without processing full sequence
- Can save 30-50% compute on average

## Key Takeaways
- **Batching + bucketing** gives the biggest immediate throughput gain with zero quality loss
- **Quantization** (INT8/FP16) is the highest-ROI model optimization — always do it
- Consider **replacing the RNN** entirely with Transformers or TCNs for production
- Use `pack_padded_sequence` in PyTorch to avoid wasting compute on padding